# 02 · Calibration, resolution and baselines

In [ ]:
import sys

sys.path.insert(0, "../src")
import matplotlib.pyplot as plt
import numpy as np

from forecaster.scoring import (
    bootstrap_ci,
    calibration_table,
    expected_calibration_error,
    murphy_decomposition,
)

## Reliability diagram

Group forecasts into bins by predicted probability; plot the average forecast in each bin against the observed frequency of YES in that bin. Points on the diagonal mean 'well calibrated.'

In [ ]:
rng = np.random.default_rng(0)
n = 2000
true_p = rng.uniform(0.05, 0.95, n)
outcomes = (rng.random(n) < true_p).astype(float)
# an overconfident forecaster: pushes every probability away from 0.5
overconfident = np.clip(0.5 + (true_p - 0.5) * 1.6, 0.02, 0.98)
table = calibration_table(overconfident, outcomes, bins=10)
xs = [r["avg_forecast"] for r in table if r["count"]]
ys = [r["observed_freq"] for r in table if r["count"]]
plt.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
plt.plot(xs, ys, "o-", label="overconfident forecaster")
plt.xlabel("average forecast in bin")
plt.ylabel("observed frequency")
plt.legend()

## Expected calibration error (ECE)

The average gap from the diagonal.

In [ ]:
print("ECE, overconfident forecaster:", expected_calibration_error(overconfident, outcomes))
print("ECE, true probabilities:", expected_calibration_error(true_p, outcomes))

## Murphy decomposition

`Brier = reliability - resolution + uncertainty`. Reliability is miscalibration (lower is better). Resolution is how well the forecaster separates YES from NO cases (higher is better). Uncertainty is fixed by how unpredictable the questions are — no forecaster changes it.

In [ ]:
decomp = murphy_decomposition(overconfident, outcomes, bins=10)
decomp

## Why the base-rate forecaster is calibrated but useless

A forecaster that always reports the overall base rate is *perfectly calibrated* on average — but it has zero resolution, because it never distinguishes a likely YES from a likely NO.

In [ ]:
base_rate = outcomes.mean()
base_rate_forecast = np.full(n, base_rate)
murphy_decomposition(base_rate_forecast, outcomes, bins=10)

## Bootstrap confidence intervals

With a small number of resolved questions, a real-looking difference in scores can easily be noise.

In [ ]:
def mean_stat(v):
    return float(np.mean(v))


small_sample = outcomes[:60]
large_sample = outcomes[:600] if n >= 600 else outcomes
lo_s, hi_s = bootstrap_ci(small_sample, mean_stat, n_boot=1000, seed=1)
lo_l, hi_l = bootstrap_ci(large_sample, mean_stat, n_boot=1000, seed=1)
print(f"60 questions:  [{lo_s:.3f}, {hi_s:.3f}], width={hi_s - lo_s:.3f}")
print(f"{len(large_sample)} questions: [{lo_l:.3f}, {hi_l:.3f}], width={hi_l - lo_l:.3f}")

**Try it:** raise the sample from 60 to 600 above (already done if `n >= 600`) and watch the interval narrow.